# Pricing Options with Binomial Trees: A Student Comparison with Black-Scholes

**Project type:** semi-learning paper notebook  
**Research question:** how does a discrete-time binomial tree approximate Black-Scholes, and why is it useful for American-style exercise?

## Abstract

This note studies the Cox-Ross-Rubinstein binomial tree as a practical bridge between intuitive option payoffs and continuous-time Black-Scholes pricing. The notebook implements a recombining tree, compares European option prices against the Black-Scholes formula, and studies the early-exercise premium for American puts. My aim is to understand why a simple up/down model can still be powerful: it is visual, programmable, and flexible enough to handle exercise features that are harder to express in closed form.

## Personal Learning Position

Black-Scholes feels elegant but abstract. The binomial tree feels more mechanical: at each time step the stock goes up or down, and the option value is found by working backward. I like this model because it makes replication and risk-neutral pricing feel more concrete. It also gives me a way to see how a continuous-time formula can emerge from many small discrete steps.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate / "src"))
        PROJECT_ROOT = candidate
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from options_pricing_research.binomial import cox_ross_rubinstein_price, terminal_stock_tree
from options_pricing_research.black_scholes import black_scholes_price

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:.4f}".format

## 1. Model Setup

The Cox-Ross-Rubinstein tree assumes that over a small time interval the stock price moves to either $S u$ or $S d$, where:

$$
u = e^{\sigma\sqrt{\Delta t}}, \qquad d = \frac{1}{u}.
$$

The risk-neutral probability is:

$$
p = \frac{e^{(r-q)\Delta t} - d}{u - d}.
$$

The option is priced backward from terminal payoffs. For American options, the model compares continuation value with immediate exercise value at every node.

## 2. A Small Tree for Intuition

A small tree is not accurate, but it is useful for learning. It shows how the same starting price fans out into possible future states.

In [ ]:
small_tree = terminal_stock_tree(spot=100.0, time_to_expiry=1.0, volatility=0.20, steps=3)
pd.DataFrame({f"step_{i}": pd.Series(row) for i, row in enumerate(small_tree)})

My first impression is that the binomial tree makes uncertainty more tangible. Instead of immediately jumping to a normal distribution or stochastic differential equation, I can see the possible stock prices as a branching structure. This is helpful before moving back to continuous-time models.

## 3. Convergence to Black-Scholes

If the tree is constructed correctly, a European option price should approach the Black-Scholes price as the number of steps increases.

In [ ]:
spot = 100.0
strike = 100.0
time_to_expiry = 1.0
risk_free_rate = 0.05
volatility = 0.20

bs_price = black_scholes_price("call", spot, strike, time_to_expiry, risk_free_rate, volatility)
steps_grid = [5, 10, 25, 50, 100, 250, 500, 1_000]
convergence = pd.DataFrame(
    {
        "steps": steps_grid,
        "binomial_call_price": [
            cox_ross_rubinstein_price("call", spot, strike, time_to_expiry, risk_free_rate, volatility, steps=steps)
            for steps in steps_grid
        ],
    }
)
convergence["black_scholes_price"] = bs_price
convergence["absolute_error"] = (convergence["binomial_call_price"] - bs_price).abs()
convergence

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(convergence["steps"], convergence["binomial_call_price"], marker="o", label="CRR tree")
ax.axhline(bs_price, color="black", linestyle="--", label="Black-Scholes")
ax.set_xscale("log")
ax.set_title("Binomial Tree Convergence to Black-Scholes")
ax.set_xlabel("Tree steps, log scale")
ax.set_ylabel("European call price")
ax.legend()
plt.show()

The convergence is the main reason this model feels satisfying to learn. The binomial tree starts as a simple discrete model, but with enough steps it moves toward the continuous Black-Scholes benchmark. My takeaway is that numerical methods are not separate from theory; they can approximate theory when closed-form methods are available and extend it when they are not.

## 4. American Exercise

The Black-Scholes formula in the first paper prices European options. A binomial tree can also price American options because it checks at every node whether early exercise is more valuable than continuation.

In [ ]:
strike_grid = [80, 90, 100, 110, 120]
american_table = pd.DataFrame(
    [
        {
            "strike": current_strike,
            "european_put": cox_ross_rubinstein_price("put", spot, current_strike, time_to_expiry, risk_free_rate, volatility, steps=500, exercise_style="european"),
            "american_put": cox_ross_rubinstein_price("put", spot, current_strike, time_to_expiry, risk_free_rate, volatility, steps=500, exercise_style="american"),
        }
        for current_strike in strike_grid
    ]
)
american_table["early_exercise_premium"] = american_table["american_put"] - american_table["european_put"]
american_table

This section helped me understand why model choice depends on contract design. If the option allows early exercise, a closed-form European formula may be the wrong tool. The binomial tree is less elegant than Black-Scholes, but it is more flexible in this setting.

## 5. Reflection

My view after building this notebook is that the binomial tree is a good second model after Black-Scholes. It is simple enough to explain, but powerful enough to introduce numerical pricing, convergence, and American exercise. It also teaches an important habit for quantitative finance: sometimes a model is valuable not because it is the most mathematically compact, but because it adapts well to the product being priced.

## References

- Cox, J. C., Ross, S. A., and Rubinstein, M. (1979). Option Pricing: A Simplified Approach. *Journal of Financial Economics*.
- Hull, J. C. *Options, Futures, and Other Derivatives*.